# 03 — Pre-registered analysis (Phase 5b)

**Did a commitment statement improve HINTS 7 data quality?**

This notebook is the narrative for the Phase 5b re-run. It supersedes the withdrawn Phase 5 analysis, which computed correct per-respondent point estimates and then computed variance as though every survey *item* were an independent observation — making every confidence interval about 21× too narrow and turning two nulls into large, highly significant effects.

The rule obeyed everywhere here: **every estimate is a weighted mean of a per-respondent rate. The unit of analysis is the respondent — 7,278 numbers, not ~2 million.** Every estimate routes through `src/weighting.py` unmodified; `estimate_rate()` carries a wired-in assertion that fails if a standard error implies more independent observations than there are respondents. It was never suppressed and never fired on a pre-registered estimate.

Numbers, tables, and figures are produced by `src/analysis.py` and written to `outputs/`. This notebook explains them; run `python src/analysis.py` to regenerate.

---

## Headline

**The commitment statement did not significantly reduce item nonresponse, break-off, or response error at the family-wise 0.05 level.**

- **H1 (primary, item nonresponse, ITT):** treatment 1.22% vs control 1.44%, difference **−0.21 pp, 95% CI [−0.44, +0.01] pp, z = −1.86, p = 0.063 uncorrected (p_Holm = 0.19).** Not significant. The direction (a reduction) is the one H1 predicted — it is simply not distinguishable from zero.
- **H2 (break-off, web only):** +1.82 pp, 95% CI [−1.17, +4.80], p = 0.23. Null.
- **H3 (response error):** −0.02 pp, 95% CI [−0.10, +0.06], p = 0.62. Null.
- The withdrawn Phase 5 reported H1 z = −13.14 (p < 0.0001) and H3 z = −3.53 (p = 0.0071). Neither survives respondent-level variance.

**Is the H1 null informative?** Yes. The observed effect (0.21 pp) is smaller than the empirical MDE (0.32 pp) and far below the pre-registration grid's smallest MDE (1.8 pp). By the pre-registered rule (effect < MDE and MDE < 3 pp), this is an **informative null**: an item-nonresponse effect of the size the plan was built to detect is ruled out.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import analysis as A

frame = pd.read_parquet(A.FRAME_PATH)
frame.shape

## 1–3. Pre-registered ITT analyses (run first, recorded before anything else)

Treatment arm = `Treatment_H7_2 == "Included in Commitment Statement group"` (n = 1,513). Control = everyone not assigned (n = 5,765) — the pre-registered ITT comparison group, which is unbiased because assignment was randomised and nobody outside the statement arm saw the statement.

Point estimate: weighted mean of the per-respondent rate, `PERSON_FINWT0`. Variance: NCI jackknife over the 50 replicate weights, `Var = 0.98 · Σ(θᵢ − θ₀)²` (no `/50`). SE of the difference: the pre-registered form `√(SE_tx² + SE_ctl²)`. Two-sided p from the normal tail, independently re-checked against `χ²₁`.

In [ ]:
res = A.run_primary_and_secondary(frame)
A._primary_table(res)

In [ ]:
for k in ['H1', 'H2', 'H3']:
    print(res[k]['tag'], '->', res[k]['verdict'])

### The MDE reckoning (risk R4)

The pre-registration's MDE grid assumed a design effect of 1.2–1.5 and baseline rates of 5–25%. Phase 4b **measured** the design effect for the per-respondent rate estimator: **1.56 (Family A), 1.44 (C), 2.93 (B)** — confirmed in R `survey`. The review's "~3.3" was the Kish weight design effect, not the estimator's.

Two MDE numbers are reported for each hypothesis:

- **grid formula** — the pre-registration's own `2.8 · √(p(1−p)(1/nₜ + 1/nᴄ)) · √DEFF`, evaluated at the *observed* baseline with the *measured* DEFF substituted. This is the grid applied, not re-reasoned.
- **empirical** — `2.8 · SE(difference)` from the actual arm jackknife SEs. This already embeds the true dispersion and design effect and is the honest per-arm MDE.

For Family A the realised rate (~1.4%) is far below the grid's lowest assumed baseline (5%) and the rate's weighted variance is ~9× below `p(1−p)`, so **the grid is conservative, not optimistic**. The experiment was *better* powered for the primary outcome than the locked plan claimed. The `"experiment may not have been powered to answer its own question"` framing applies to **H2 (break-off)** — empirical MDE 4.3 pp against plausible 1–3 pp effects — and, in proportional terms, to the very rare **H3** outcome; it does **not** apply to the primary outcome H1.

In [ ]:
A._mde_table(res)

## 4. Mode subgroup (pre-registered)

The one pre-registered subgroup. Its pre-registered question is whether the treatment effect *differs* by mode — the interaction (web effect minus paper effect). Both interactions are null (A: +0.62 pp, p = 0.057; C: +0.08 pp, p = 0.60).

The paper-only Family A cell shows a nominal reduction (−0.67 pp, p ≈ 0.03 uncorrected). It is **not** elevated to a finding: it does not survive multiplicity, the interaction that would license a mode-specific claim is null, and the pre-registration explicitly forbids hunting a subgroup that moved once the pre-registered tests are null. It is reported in the table and left there.

**Collider caveat (Neyda's ruling, 2026-09-03).** Survey mode is measured *after* randomisation and is associated with arm in the responding sample (§8: 70.1% vs 65.9% web, p = 0.002). Conditioning on a post-treatment variable can open a non-causal arm→outcome path (collider / selection bias), so the mode-stratified cells are **descriptive supplements**, not de-confounded within-mode causal effects. The unbiased estimand stays the pooled ITT contrast, which does not condition on mode. The pre-registered use of this subgroup — the interaction test — is null and is unaffected by the caveat.


In [ ]:
mode_df = A.run_mode_subgroup(frame)
mode_df

## 5. Multiplicity — Holm-Bonferroni

Applied exactly as written: rank p-values ascending, compare the k-th smallest to `alpha / (m − k + 1)`, step down.

- **m = 3** — the three outcome-family tests. This is what the pre-registration's numbered decision rule specifies (`alpha/3, alpha/2, alpha/1`).
- **m = 5** — supplementary, adding the two pre-registered mode-interaction tests, since the plan also says "any subgroup analyses … are also in the family."

Nothing is significant before correction under either reading, so Holm changes no verdict. The correction family is fixed by the plan, not by how many analyses completed — unlike the withdrawn Phase 5, which corrected over the two tests it managed to run.

In [ ]:
holm = A.run_multiplicity(res, mode_df)
print('m = 3:'); display(holm['holm_m3'])
print('m = 5 (supplementary):'); display(holm['holm_m5'])

## 6. Per-protocol (NON-RANDOMISED — descriptive only, not causal)

Among the 1,513 assigned to the statement arm: 1,389 agreed (`CommitmentStmt == "Yes"`), 124 did not (14 "No" + 110 "Not Ascertained"). `CommitmentStmt` is intact — the withdrawn Phase 5's n = 0 was a filtering bug (the values are exact codebook strings, and `Treatment_H7_2` is categorical text, not `1`/`2`).

Those who agreed show *lower* item nonresponse than those who did not (1.17% vs 1.86%; descriptive difference −0.69 pp). **This is not causal.** Agreement is confounded with engagement, education and motivation, which independently predict data quality. The label "NON-RANDOMISED" is repeated in the table, the figure caption, and this sentence because a single footnote is not enough (risk R7). The 14 who declined are described, not tested.

In [ ]:
A.run_per_protocol(frame)

## 7. Sensitivity specifications

### 7a. Filter-Missing rule (pre-registered dual specification)

The pre-registration dual-specifies the `Missing data (Filter Missing)` rule: primary includes those items in the denominator, sensitivity excludes them.

Primary and sensitivity agree in **sign and magnitude** for all three families. For H1 the uncorrected p-values straddle 0.05 (primary 0.063, sensitivity 0.025; point estimates −0.21 vs −0.27 pp), but **neither survives the pre-registered Holm correction** (H1 p_Holm = 0.19 primary; 3 × 0.025 = 0.076 sensitivity). The qualitative verdict — null — does not depend on the Filter-Missing coding decision; only the distance to the 0.05 line moves.

### 7b. Web-Never-Seen in Family A's denominator (footnote; specification NOT changed)

The pre-registration's Family A denominator table says `Missing data (Web partial - Question Never Seen)` cells should not be counted; the build (frozen since Phase 3) includes them (107,773 cells across 612 respondents — recomputed here from the raw `.rda`, matching Phase 3b/4b). **Neyda ruled (2026-09-03) to keep the as-built specification** — changing an outcome definition after seeing results is the move this project exists to avoid — and to report the alternative as a footnote sensitivity.

Excluding those cells (the plan's literal rule) shrinks the H1 point estimate from −0.213 pp to −0.097 pp and moves p from 0.063 to 0.55. Because the treatment arm is more web-heavy, it carries proportionally more web-only cells; excluding them lifts its denominator more and narrows the arm gap. **The direction is unchanged and the null becomes _more_ clearly a null, not less.** H3 is essentially unchanged; H2 is not applicable (Family B *is* the Web-Never-Seen count). The `family_a_denominator` / `family_a_rate` columns are byte-for-byte what Phase 3 wrote.


In [ ]:
sens = A.run_sensitivity(frame, res)
print('7a Filter-Missing:'); display(sens['filter_missing'])
print('7b Web-Never-Seen (spec NOT changed):'); display(sens['web_never_seen'])

## 8. Covariate balance (deferred from Phase 1)

The pre-registration did **not** enumerate balance covariates — see “What the pre-registration got wrong” below. A standard demographic set plus survey mode and the design stratum is used, unweighted Pearson χ² of arm × covariate independence.

**How to read this (Neyda's ruling, 2026-09-03 — Phase 1 is NOT reopened).** A balance check on a survey experiment is **conditional on response**: it compares the arms among the ~27% who answered, not among those randomised. Every covariate here (survey mode included) is measured at or after response — **post-randomisation**. A gap is not evidence that randomisation failed, and a balance test cannot overturn what Phase 1 established by an independent route (the codebook label + `CommitmentStmt` alignment).

**Seven of eight covariates are balanced**, including every demographic and the design stratum (`STRATUM` p = 0.43 — despite the methodology report's stratified-allocation language, the realised split does not differ by stratum). **Survey mode is imbalanced**: the treatment arm is 70.1% web vs 65.9% in control (χ² p = 0.002, 4.2 pp gap, survives a Bonferroni over the eight). This is **a finding to report, not a defect to fix**: the responding samples differ slightly on one post-treatment variable. It does not bias the pooled ITT estimand (which does not condition on mode), and the mode-stratified analysis (§4, with its collider caveat) is null within each mode.


In [ ]:
A.run_balance(frame)

## 9. Peeking illustration (NOT evidence)

A demonstration of why the plan was locked. No arrival-order or sequence field exists in the public file, so respondents are put in a **seeded random order** (seed 20260903) and the H1 ITT p-value is recomputed as the sample accrues. The trajectory wanders between ~0.06 and ~1.0 across peeks and lands at the final 0.063. It is not an inference about the hypothesis; it is a picture of the noise that repeated looking would have exposed you to.

In [ ]:
A.run_peeking(frame)

## Assertion history — the `implied_n` guard

`estimate_rate()` runs `assert_variance_plausible()` on every estimate: it fails if `dispersion / se²` exceeds the respondent count (with a 1.10 cushion). This is the guard built in Phase 4b specifically to stop the Phase 5 failure mode. It ran on **all 27 pre-registered estimates and never fired**; `implied_n / n` ranged 0.15–0.79. It did not fire in the peeking illustration either. Had the withdrawn Phase 5's item-level variance been computed here, it would have fired at `implied_n / n ≈ 22`.

In [ ]:
pd.DataFrame(A._ASSERTION_LOG)

## Independent cross-checks

The exact quantities that failed in Phase 5 — the arm-split rates and their standard errors — were reproduced three ways and agree to six decimal places:

| quantity | `weighting.py` | from-scratch numpy | R `survey` (JKn, scale 0.98) |
|---|---|---|---|
| H1 treatment SE | 0.093495 pp | 0.093495 pp | 0.093495 pp |
| H1 control SE | 0.066238 pp | 0.066238 pp | 0.066238 pp |
| H1 difference z | −1.85796 | −1.85796 | — |

See `docs/validation.md` (Phase 5b section) for the full table.

## What the pre-registration got wrong

1. **The MDE grid is conservative, not what it claimed.** It assumed 5–25% baseline rates and `p(1−p)` per-respondent variance. Item nonresponse is ~1.4% and its rate variance is ~9× smaller than `p(1−p)`, so the true primary-outcome MDE (~0.3–0.4 pp) is 4–10× below the grid's 1.8–4.0 pp. The plan's own null-interpretation rule still resolves cleanly (informative null), but the grid understated the experiment's power. **Recorded as a dated `Amendment 1` to `docs/pre-registration.md` (Neyda-approved, 2026-09-03; original text untouched)**, which also notes the “≈ 3.3” design effect was the Kish weight DEFF, not the estimator's (~1.5).
2. **No balance covariates were specified.** Part 4 of the analysis prompt requires a balance check; the plan named no covariate list, so the set used here is a documented post-hoc choice and its p-values are not in any multiplicity family.
3. **The multiplicity family is described two ways.** "Total number of tests … (before subgroups): 3" and a decision rule written for m = 3, versus "any subgroup analyses … are also in the family." Both readings are reported (m = 3 and m = 5); neither changes a verdict here.
4. **Break-off scope for paper respondents** was under-specified for a per-respondent frame (resolved as NaN in Phase 3b; documented).

## What was not run because it was not pre-registered

No demographic, engagement, or item-level subgroup search. No mode-standardised or covariate-adjusted ITT (the mode-stratified pre-registered subgroup already shows null within each mode). No alternative outcome definitions beyond the dual-specified Filter-Missing rule. The Family A / Web-Never-Seen denominator question raised in Phase 3b/4b is **resolved** (Neyda, 2026-09-03): keep the as-built specification, report the alternative as the §7b footnote sensitivity. An outcome definition is not changed after seeing results.